# Analise Exploratoria de Dados - Cancer de Utero

Este notebook realiza uma primeira Analise Exploratoria de Dados (EDA) da base em Excel colocada em `data/raw`. A ideia e entender a estrutura da planilha, verificar problemas iniciais de qualidade, observar a distribuicao das variaveis principais e analisar o alvo `label_cid`, que separa os registros por categoria CID.

As quatro primeiras secoes seguem a mesma logica do notebook de referencia:

- carregamento dos dados;
- verificacao inicial da qualidade da base;
- estatisticas descritivas gerais;
- analise inicial do alvo.


## 1. Carregamento dos dados

Nesta etapa carregamos a planilha bruta, verificamos as abas disponiveis e escolhemos automaticamente a primeira aba com dados. Tambem definimos algumas colunas centrais para a analise inicial.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 80)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")


In [ ]:
raw_dir_candidates = [Path("../data/raw"), Path("data/raw")]
raw_dir = next((path for path in raw_dir_candidates if path.exists()), raw_dir_candidates[0])

xlsx_files = sorted(raw_dir.glob("*.xlsx"))
if not xlsx_files:
    raise FileNotFoundError(f"Nenhum arquivo .xlsx encontrado em {raw_dir.resolve()}")

data_path = xlsx_files[0]
excel_file = pd.ExcelFile(data_path)
sheet_shapes = []

for sheet in excel_file.sheet_names:
    preview = pd.read_excel(data_path, sheet_name=sheet, nrows=5)
    sheet_shapes.append({"aba": sheet, "colunas_preview": preview.shape[1], "tem_dados_preview": preview.shape[0] > 0})

sheet_summary = pd.DataFrame(sheet_shapes)
display(sheet_summary)

sheet_name = sheet_summary.loc[sheet_summary["tem_dados_preview"], "aba"].iloc[0]
df = pd.read_excel(data_path, sheet_name=sheet_name)

print(f"Arquivo carregado: {data_path}")
print(f"Aba utilizada: {sheet_name}")
print(f"Shape da base: {df.shape[0]:,} linhas x {df.shape[1]:,} colunas")
print("\nColunas:")
print(df.columns.tolist())


In [ ]:
target_col = "label_cid"
age_col = "idade_obito_anos"
death_date_col = "DTOBITO"
birth_date_col = "DTNASC"

region_cols = [col for col in ["res_REGIAO", "ocor_REGIAO"] if col in df.columns]
uf_cols = [col for col in ["res_SIGLA_UF", "ocor_SIGLA_UF"] if col in df.columns]
cause_cols = [
    col
    for col in ["CAUSABAS", "causabas_categoria", "causabas_subcategoria", "CB_PRE", target_col]
    if col in df.columns
]
selected_context_cols = [col for col in [age_col, death_date_col, birth_date_col, target_col] if col in df.columns]

display(df.head())
display(Markdown("### Colunas centrais identificadas"))
display(pd.DataFrame({
    "grupo": ["alvo", "idade", "datas", "regiao", "uf", "causas"],
    "colunas": [
        [target_col] if target_col in df.columns else [],
        [age_col] if age_col in df.columns else [],
        [col for col in [death_date_col, birth_date_col] if col in df.columns],
        region_cols,
        uf_cols,
        cause_cols,
    ],
}))


## 2. Verificacao inicial da qualidade da base

Antes de discutir distribuicoes, e importante verificar se a base apresenta problemas estruturais. Nesta secao analisamos:

- valores nulos;
- tipos de dados;
- duplicatas;
- valores infinitos em colunas numericas;
- colunas constantes ou quase constantes;
- inconsistencias simples em idade e datas;
- codigos de ausencia ou ignorado em variaveis categoricas/codificadas.

Essas verificacoes ajudam a separar problemas reais de qualidade de dados de caracteristicas esperadas em bases administrativas de mortalidade.


In [ ]:
def build_count_percent_table(series: pd.Series, total_rows: int, count_name: str = "count") -> pd.DataFrame:
    summary = pd.DataFrame({count_name: series})
    summary["percent"] = 100 * summary[count_name] / total_rows
    return summary


def display_section(title: str) -> None:
    display(Markdown(f"### {title}"))


def parse_yyyymmdd(series: pd.Series) -> pd.Series:
    values = series.copy()
    values = values.where(values.notna(), np.nan)
    values = values.astype("Int64").astype(str).replace("<NA>", np.nan)
    return pd.to_datetime(values, format="%Y%m%d", errors="coerce")


In [ ]:
n_rows = len(df)
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()

null_summary = build_count_percent_table(df.isna().sum(), n_rows, count_name="null_count")
dtype_summary = pd.DataFrame({"dtype": df.dtypes.astype(str), "unique_values": df.nunique(dropna=True)})
inf_summary = build_count_percent_table(
    pd.Series(np.isinf(df[numeric_cols]).sum(), index=numeric_cols) if numeric_cols else pd.Series(dtype=int),
    n_rows,
    count_name="inf_count",
)

duplicate_summary = pd.DataFrame({"count": [int(df.duplicated().sum())]}, index=["duplicated_rows"])
duplicate_summary["percent"] = 100 * duplicate_summary["count"] / n_rows

constant_cols = dtype_summary.query("unique_values <= 1").copy()
near_constant_cols = dtype_summary.query("unique_values > 1 and unique_values <= 3").copy()

display_section("Resumo geral")
display(pd.DataFrame({
    "indicador": ["linhas", "colunas", "colunas_numericas", "colunas_categoricas"],
    "valor": [n_rows, df.shape[1], len(numeric_cols), len(categorical_cols)],
}))

display_section("Valores nulos por coluna")
display(null_summary.sort_values(["null_count", "percent"], ascending=False))

display_section("Tipos de dados e cardinalidade")
display(dtype_summary.sort_values("unique_values", ascending=False))

display_section("Valores infinitos por coluna numerica")
display(inf_summary.sort_values("inf_count", ascending=False))

display_section("Duplicatas")
display(duplicate_summary)

display_section("Colunas constantes")
display(constant_cols if not constant_cols.empty else pd.DataFrame({"mensagem": ["Nenhuma coluna constante encontrada."]}))

display_section("Colunas com ate 3 valores distintos")
display(near_constant_cols)


### Verificacoes de idade e datas

A idade ao obito e as datas sao variaveis importantes para entender a consistencia temporal da base. Aqui convertemos datas no padrao `AAAAMMDD`, quando possivel, apenas em memoria.


In [ ]:
quality_checks = {}

if age_col in df.columns:
    quality_checks["idade_nula"] = df[age_col].isna().sum()
    quality_checks["idade_negativa"] = (df[age_col] < 0).sum()
    quality_checks["idade_maior_120"] = (df[age_col] > 120).sum()

date_quality = []
date_columns = [col for col in [death_date_col, birth_date_col] if col in df.columns]

for col in date_columns:
    parsed = parse_yyyymmdd(df[col])
    date_quality.append({
        "coluna": col,
        "datas_validas": parsed.notna().sum(),
        "datas_invalidas_ou_nulas": parsed.isna().sum(),
        "min": parsed.min(),
        "max": parsed.max(),
    })

if all(col in df.columns for col in [death_date_col, birth_date_col]):
    death_dates = parse_yyyymmdd(df[death_date_col])
    birth_dates = parse_yyyymmdd(df[birth_date_col])
    quality_checks["nascimento_depois_do_obito"] = (birth_dates > death_dates).sum()

display_section("Checagens simples de idade e consistencia temporal")
display(build_count_percent_table(pd.Series(quality_checks), n_rows) if quality_checks else pd.DataFrame())

display_section("Resumo de conversao das datas")
display(pd.DataFrame(date_quality))


### Codigos de ausencia, ignorado ou nao informado

Bases do SIM e bases administrativas frequentemente usam codigos como `9`, `99`, `999` ou textos como `IGNORADO`. Esta tabela nao remove nada; ela apenas sinaliza onde pode haver categorias que precisam de tratamento no pre-processamento.


In [ ]:
missing_like_tokens = {"", " ", "9", "99", "999", "9999", "IGN", "IGNORADO", "NAO INFORMADO"}
missing_like_summary = []

for col in df.columns:
    values_as_text = df[col].astype("string").str.strip().str.upper()
    count = values_as_text.isin(missing_like_tokens).sum()
    if count > 0:
        missing_like_summary.append({
            "coluna": col,
            "missing_like_count": int(count),
            "percent": 100 * count / n_rows,
        })

missing_like_summary = pd.DataFrame(missing_like_summary)
display(missing_like_summary.sort_values("missing_like_count", ascending=False) if not missing_like_summary.empty else pd.DataFrame({"mensagem": ["Nenhum codigo desse tipo foi encontrado."]}))


### Valores extremos em variaveis continuas

A regra do IQR e usada aqui apenas para sinalizar observacoes atipicas. Como varias colunas numericas sao codigos categoricos, aplicamos essa leitura inicialmente em variaveis com interpretacao continua, como idade e intervalos em dias.


In [ ]:
continuous_cols = [col for col in [age_col, "NUDIASOBCO", "NUDIASINF"] if col in df.columns]

if continuous_cols:
    q1 = df[continuous_cols].quantile(0.25)
    q3 = df[continuous_cols].quantile(0.75)
    iqr = q3 - q1
    lower_fence = q1 - 1.5 * iqr
    upper_fence = q3 + 1.5 * iqr

    extreme_summary = pd.DataFrame(index=continuous_cols)
    extreme_summary["lower_fence"] = lower_fence
    extreme_summary["upper_fence"] = upper_fence
    extreme_summary["below_lower_fence"] = [int((df[col] < lower_fence[col]).sum()) for col in continuous_cols]
    extreme_summary["above_upper_fence"] = [int((df[col] > upper_fence[col]).sum()) for col in continuous_cols]
    extreme_summary["total_extreme"] = extreme_summary["below_lower_fence"] + extreme_summary["above_upper_fence"]
    extreme_summary["percent_extreme"] = 100 * extreme_summary["total_extreme"] / n_rows

    display(extreme_summary.sort_values("percent_extreme", ascending=False))
else:
    display(pd.DataFrame({"mensagem": ["Nenhuma coluna continua pre-definida foi encontrada."]}))


### Leituras iniciais da qualidade da base

Depois de executar as tabelas acima, observe principalmente:

- quais colunas tem muitos valores nulos;
- quais colunas sao codigos categoricos, mesmo aparecendo como numericas;
- se `idade_obito_anos` tem valores fora de uma faixa plausivel;
- se as datas foram convertidas corretamente;
- se existem colunas constantes que nao devem entrar no modelo;
- se codigos como `9`, `99` e `IGNORADO` devem virar uma categoria explicita ou valor ausente no pre-processamento.

Essa leitura e especialmente importante antes de aplicar rede neural, porque a rede vai exigir codificacao adequada das variaveis categoricas e padronizacao das variaveis numericas continuas.


## 3. Estatisticas descritivas gerais

Nesta etapa analisamos a escala das variaveis, a cardinalidade das colunas categoricas e algumas distribuicoes iniciais. O objetivo e identificar o que precisara de tratamento antes do algoritmo genetico e da rede neural.


### Visao tabular geral


In [ ]:
display(df[selected_context_cols + region_cols + uf_cols + cause_cols].head())
display(df.describe(include="all").T)


### Escalas das variaveis numericas

A tabela abaixo ajuda a distinguir variaveis realmente continuas de codigos numericos. Em redes neurais, variaveis continuas costumam precisar de padronizacao, enquanto codigos categoricos precisam de codificacao apropriada.


In [ ]:
numeric_scale_summary = df[numeric_cols].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).T
numeric_scale_summary["iqr"] = numeric_scale_summary["75%"] - numeric_scale_summary["25%"]
numeric_scale_summary["missing_percent"] = 100 * df[numeric_cols].isna().sum() / n_rows
display(numeric_scale_summary.sort_values("missing_percent", ascending=False))


### Cardinalidade e categorias mais frequentes

Aqui vemos quantos valores distintos existem em cada variavel categorica e quais categorias aparecem com maior frequencia nas principais colunas textuais.


In [ ]:
categorical_summary = pd.DataFrame({
    "unique_values": df[categorical_cols].nunique(dropna=True),
    "missing_percent": 100 * df[categorical_cols].isna().sum() / n_rows,
}).sort_values("unique_values", ascending=False)

display(categorical_summary)

top_category_cols = [col for col in [target_col, "CAUSABAS", "causabas_categoria", "causabas_subcategoria", "res_REGIAO", "ocor_REGIAO", "res_SIGLA_UF", "ocor_SIGLA_UF"] if col in df.columns]

for col in top_category_cols:
    display_section(f"Top categorias - {col}")
    top_values = df[col].value_counts(dropna=False).head(15).rename_axis(col).reset_index(name="frequencia")
    top_values["percent"] = 100 * top_values["frequencia"] / n_rows
    display(top_values)


### Distribuicoes iniciais


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

if age_col in df.columns:
    sns.histplot(df[age_col].dropna(), bins=40, kde=True, ax=axes[0], color="#2a6f97")
    axes[0].set_title("Distribuicao da idade ao obito")
    axes[0].set_xlabel("idade ao obito em anos")

    sns.boxplot(x=df[age_col], ax=axes[1], color="#89c2d9")
    axes[1].set_title("Boxplot da idade ao obito")
    axes[1].set_xlabel("idade ao obito em anos")
else:
    axes[0].text(0.5, 0.5, "Coluna de idade nao encontrada", ha="center")
    axes[1].axis("off")

plt.tight_layout()
plt.show()


In [ ]:
plot_cols = [col for col in ["res_REGIAO", "ocor_REGIAO", "res_SIGLA_UF", "ocor_SIGLA_UF"] if col in df.columns]

if plot_cols:
    fig, axes = plt.subplots(len(plot_cols), 1, figsize=(12, max(4, 3.8 * len(plot_cols))))
    axes = np.atleast_1d(axes)

    for ax, col in zip(axes, plot_cols):
        counts = df[col].value_counts(dropna=False).head(15).reset_index()
        counts.columns = [col, "frequencia"]
        sns.barplot(data=counts, y=col, x="frequencia", ax=ax, color="#2a6f97")
        ax.set_title(f"Categorias mais frequentes - {col}")
        ax.set_xlabel("frequencia")
        ax.set_ylabel("")

    plt.tight_layout()
    plt.show()


### Leitura das estatisticas gerais

Nesta base, e importante nao tratar toda coluna numerica como medida continua. Muitas variaveis sao codigos administrativos, como sexo, raca/cor, escolaridade, local de ocorrencia e municipio. Para modelagem com rede neural, isso sugere um pre-processamento com separacao clara entre:

- variaveis continuas, como `idade_obito_anos`;
- variaveis categoricas codificadas por numero;
- variaveis textuais de causa e localizacao;
- colunas com muitos nulos ou cardinalidade muito alta, que podem exigir selecao, agrupamento ou descarte.


## 4. Analise do alvo `label_cid`

Nesta secao exploramos o alvo inicial do problema. A coluna `label_cid` indica a categoria CID associada ao registro, e pode ser usada como alvo em uma tarefa de classificacao. O objetivo aqui e verificar desbalanceamento entre classes e observar relacoes iniciais com idade e localizacao.


### Distribuicao das classes


In [ ]:
target_summary = df[target_col].value_counts(dropna=False).rename_axis(target_col).reset_index(name="frequencia")
target_summary["percent"] = 100 * target_summary["frequencia"] / n_rows
target_summary["percent_acumulado"] = target_summary["percent"].cumsum()

display(target_summary.style.format({
    "frequencia": "{:,}",
    "percent": "{:.2f}%",
    "percent_acumulado": "{:.2f}%",
}))


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))

sns.barplot(data=target_summary, x=target_col, y="percent", ax=ax, color="#2a6f97")
ax.set_title("Distribuicao percentual do alvo label_cid")
ax.set_xlabel("classe CID")
ax.set_ylabel("percentual")

for container in ax.containers:
    ax.bar_label(container, fmt="%.1f%%", padding=3)

plt.tight_layout()
plt.show()


### Idade por classe do alvo

A distribuicao da idade por classe ajuda a verificar se alguma categoria CID aparece mais associada a faixas etarias especificas. Isso e relevante porque `idade_obito_anos` tende a ser uma variavel preditiva forte em bases de mortalidade.


In [ ]:
if age_col in df.columns:
    age_by_target = (
        df.groupby(target_col)[age_col]
        .describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9])
        .sort_index()
    )
    display(age_by_target)

    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

    sns.boxplot(data=df, x=target_col, y=age_col, ax=axes[0], color="#89c2d9")
    axes[0].set_title("Idade ao obito por classe CID")
    axes[0].set_xlabel("classe CID")
    axes[0].set_ylabel("idade ao obito")

    sns.histplot(data=df, x=age_col, hue=target_col, bins=40, kde=True, stat="density", common_norm=False, ax=axes[1])
    axes[1].set_title("Distribuicao da idade por classe CID")
    axes[1].set_xlabel("idade ao obito")
    axes[1].set_ylabel("densidade")

    plt.tight_layout()
    plt.show()
else:
    display(pd.DataFrame({"mensagem": ["Coluna de idade nao encontrada."]}))


### Alvo por regiao e UF

As tabelas abaixo mostram a composicao percentual do alvo dentro das principais variaveis de localizacao. Isso ajuda a identificar diferencas regionais e possiveis efeitos de distribuicao da base.


In [ ]:
location_cols = [col for col in ["res_REGIAO", "ocor_REGIAO", "res_SIGLA_UF", "ocor_SIGLA_UF"] if col in df.columns]

for col in location_cols:
    display_section(f"Distribuicao percentual de {target_col} por {col}")
    table = pd.crosstab(df[col], df[target_col], normalize="index") * 100
    counts = df[col].value_counts(dropna=False).rename("total_registros")
    table = table.join(counts)
    display(table.sort_values("total_registros", ascending=False).head(20).style.format("{:.2f}"))


In [ ]:
region_col = "res_REGIAO" if "res_REGIAO" in df.columns else ("ocor_REGIAO" if "ocor_REGIAO" in df.columns else None)

if region_col is not None:
    region_target = (
        pd.crosstab(df[region_col], df[target_col], normalize="index")
        .mul(100)
        .reset_index()
        .melt(id_vars=region_col, var_name=target_col, value_name="percent")
    )

    fig, ax = plt.subplots(figsize=(12, 5))
    sns.barplot(data=region_target, x=region_col, y="percent", hue=target_col, ax=ax)
    ax.set_title(f"Composicao percentual de {target_col} por {region_col}")
    ax.set_xlabel(region_col)
    ax.set_ylabel("percentual dentro da regiao")
    ax.tick_params(axis="x", rotation=30)
    plt.tight_layout()
    plt.show()


### Leitura inicial do alvo

Depois de executar esta secao, observe se uma classe domina muito a distribuicao de `label_cid`. Caso exista desbalanceamento, a etapa de modelagem deve considerar metricas alem de acuracia, como `f1_macro`, matriz de confusao e desempenho por classe.

Tambem vale observar se idade, regiao ou UF parecem mudar a composicao das classes. Essas diferencas nao provam causalidade, mas indicam variaveis que podem ser relevantes para o algoritmo genetico selecionar ou para o pre-processamento tratar com mais cuidado.
